# Polydisperse Polymer Systems

**"How do I declare a molecular-weight distribution and sample a polydisperse chain ensemble in MolPy?"**

This guide demonstrates MolPy’s workflow from a **GBigSMILES** distribution declaration to a sampled chain ensemble, and then compares $M_n$ / $M_w$ / PDI against the theoretical distribution.

## What We Will Do

1. Declare four common distributions in GBigSMILES (Schulz–Zimm / Uniform / Poisson / Flory–Schulz)
2. Convert the parsed `DistributionIR` into a samplable distribution object (`create_polydisperse_from_ir`)
3. Sample a chain ensemble with `PolydisperseChainGenerator` + `SystemPlanner`
4. Compute $M_n$, $M_w$, and PDI from sampled chains and plot sample vs theory correctly (PMF vs PDF)

`SystemPlanner` (system level) repeatedly requests chains from `PolydisperseChainGenerator` (chain level) until the target total mass is met. The chain generator requests monomer sequences from a `SequenceGenerator` (sequence level).

## Setup: Imports

We will use:

- **GBigSMILES parser**: parses a string into an IR (including distribution parameters)
- **Distribution factory**: `create_polydisperse_from_ir` (turns `DistributionIR` into a samplable distribution object)
- **Chain generator**: `PolydisperseChainGenerator` (samples DP or mass, then generates a chain)
- **System planner**: `SystemPlanner` (accumulates chain mass until the target mass is met)
- **Plotting**: Matplotlib

In [ ]:
from random import Random

import matplotlib.pyplot as plt
import numpy as np
import scienceplots

plt.style.use(["science", 'nature', "no-latex"])

import molpy as mp
from molpy.builder.polymer.sequence_generator import WeightedSequenceGenerator
from molpy.builder.polymer.system import (
    FlorySchulzPolydisperse,
    PoissonPolydisperse,
    PolydisperseChainGenerator,
    SchulzZimmPolydisperse,
    SystemPlanner,
    UniformPolydisperse,
    create_polydisperse_from_ir,
 )
from molpy.core.element import Element


## Step 1: Declare Distributions in GBigSMILES

We use a compact GBigSMILES form like:

- `{repeat_units}|distribution(params)|end_group|target_mass|`

Where:

- `repeat_units`: repeat unit set (can include multiple monomers)
- `distribution(params)`: distribution type and parameters
- `end_group`: end group (we use `[H].` as a simple example)
- `target_mass`: target total mass for the system (used by `SystemPlanner.target_total_mass`)

We compare four distributions:

1. **Schulz–Zimm** (continuous, mass space): `schulz_zimm(Mn, Mw)`
2. **Uniform** (discrete, DP space): `uniform(min_dp, max_dp)`
3. **Poisson** (discrete, DP space): `poisson(mean_dp)`
4. **Flory–Schulz** (discrete, DP space): `flory_schulz(a)`
**Checkpoint:** you can parse each string and find a `DistributionIR(name, params)` in the resulting IR.


In [ ]:
def _build_distributions() -> list[dict]:
    """Configure the four distributions to test (GBigSMILES strings)."""
    return [
        {
            "name": "Schulz-Zimm",
            "gbigsmiles": "{[<]OCCOCCOCCOCCO[>],[<]OCC(c1ccccc1)CO[>]}|schulz_zimm(4800, 6000)|[H].|1e8|",
        },
        {
            "name": "Uniform",
            "gbigsmiles": "{[<]OCCOCCOCCOCCO[>],[<]OCC(c1ccccc1)CO[>]}|uniform(10,60)|[H].|1e8|",
        },
        {
            "name": "Poisson",
            "gbigsmiles": "{[<]OCCOCCOCCOCCO[>],[<]OCC(c1ccccc1)CO[>]}|poisson(30)|[H].|1e8|",
        },
        {
            "name": "Flory-Schulz",
            "gbigsmiles": "{[<]OCCOCCOCCOCCO[>],[<]OCC(c1ccccc1)CO[>]}|flory_schulz(0.06)|[H].|1e8|",
        },
    ]

## Step 2: Parse GBigSMILES and Extract Distribution Information

For each distribution, we need to:

1. Parse the GBigSMILES string to get the `DistributionIR`
2. Extract repeat units and compute monomer masses
3. Create a samplable distribution object using `create_polydisperse_from_ir`

**Checkpoint:** After parsing, you should have a list of distribution objects ready for sampling.


In [ ]:
distributions_to_test = _build_distributions()
random_seed = 43

results: list[dict] = []

for dist_config in distributions_to_test:
    # Parse GBigSMiles string
    ir = mp.parser.parse_gbigsmiles(dist_config["gbigsmiles"])

    # Extract component information
    component_ir = ir.molecules[0]
    mol_ir = component_ir.molecule
    target_mass = component_ir.target_mass

    # Extract repeat units and convert to Atomistic structures
    repeat_units = mol_ir.structure.stochastic_objects[0].repeat_units
    stoch_obj = mol_ir.structure.stochastic_objects[0]
    
    # Convert graphs to Atomistic structures to compute masses
    from molpy.parser.smiles.converter import create_monomer_from_repeat_unit
    monomer_structures = []
    for unit in repeat_units:
        monomer = create_monomer_from_repeat_unit(unit, stoch_obj)
        if monomer is not None:
            monomer_structures.append(monomer)
    
    # Compute monomer masses by summing atomic masses
    monomer_masses_list = []
    for monomer in monomer_structures:
        total_mass = 0.0
        for atom in monomer.atoms:
            element_symbol = atom.get("element") or atom.get("symbol")
            if element_symbol:
                try:
                    elem = Element(element_symbol)
                    total_mass += elem.mass
                except KeyError:
                    # Skip unknown elements
                    pass
        monomer_masses_list.append(total_mass)

    # Equal weights for all monomers (can be customized)
    weights_dict = {str(i): 1.0 for i in range(len(monomer_structures))}

    # Extract distribution IR
    distribution_ir = None
    for meta in mol_ir.stochastic_metadata:
        if meta.distribution:
            distribution_ir = meta.distribution
            break
    if distribution_ir is None:
        raise ValueError(f"No distribution IR found for {dist_config['name']}")

    # Create samplable distribution object
    dist_obj = create_polydisperse_from_ir(distribution_ir, random_seed=random_seed)

    results.append(
        {
            "name": dist_config["name"],
            "dist_obj": dist_obj,
            "weights": weights_dict,
            "monomer_masses": {str(i): m for i, m in enumerate(monomer_masses_list)},
            "target_mass": float(target_mass) if target_mass is not None else None,
        }
    )

print(f"Parsed {len(results)} distributions:")
for r in results:
    print(f"  - {r['name']}: {type(r['dist_obj']).__name__}")


## Step 3: Build Sequence Generator

The `WeightedSequenceGenerator` controls the composition of repeat units in each chain.

Here we use equal weights for all monomers, but you can customize this to control copolymer composition.


In [ ]:
# Build sequence generators for each distribution
for result in results:
    seq_gen = WeightedSequenceGenerator(monomer_weights=result["weights"])
    result["seq_generator"] = seq_gen


## Step 4: Build Chain Generator

The `PolydisperseChainGenerator` samples chain size (DP or mass) from the distribution and generates a chain sequence.

It requires:
- A sequence generator (from step 4)
- Monomer masses (to convert DP to chain mass)
- End group mass (we use 0.0 for simplicity)
- The distribution object (from step 3)


In [ ]:
# Build chain generators for each distribution
for result in results:
    chain_gen = PolydisperseChainGenerator(
        seq_generator=result["seq_generator"],
        monomer_mass=result["monomer_masses"],
        end_group_mass=0.0,
        distribution=result["dist_obj"],
    )
    result["chain_generator"] = chain_gen


## Step 5: Sample Chains with System Planner

The `SystemPlanner` accumulates chains until the target total mass is reached.

It repeatedly requests chains from the `PolydisperseChainGenerator` until:
- The accumulated mass reaches the target (within `max_rel_error`)
- Or the maximum number of iterations is reached

**Checkpoint:** After planning, you should have a list of chains with varying molecular weights and DPs.


In [ ]:
# Sample chains for each distribution
for result in results:
    target_mass = result.get("target_mass", None)
    rng = Random(random_seed)
    planner = SystemPlanner(
        chain_generator=result["chain_generator"],
        target_total_mass=target_mass if target_mass is not None else 1e7,
        max_rel_error=0.02,
    )
    system_plan = planner.plan_system(rng)
    chains = system_plan.chains

    result["molecular_weights"] = [chain.mass for chain in chains]
    result["dps"] = [chain.dp for chain in chains]
    result["n_chains"] = len(chains)

print("Sampling complete:")
for r in results:
    print(f"  - {r['name']}: {r['n_chains']} chains, total mass = {sum(r['molecular_weights']):.2e} g/mol")


## Step 6: Compute Statistics (Mn, Mw, and PDI)

From the sampled chains, we compute:

- **$M_n$** (number-average molecular weight): $\bar{M}_n = \frac{\sum M_i}{N}$
- **$M_w$** (weight-average molecular weight): $\bar{M}_w = \frac{\sum M_i^2}{\sum M_i}$
- **PDI** (polydispersity index): $\mathrm{PDI} = \frac{M_w}{M_n}$

These values can be compared against the theoretical distribution parameters.


In [ ]:
# Compute Mn, Mw, PDI from sampled chains
for result in results:
    mw_array = np.array(result["molecular_weights"], dtype=float)
    Mn = float(np.mean(mw_array))
    Mw = float(np.sum(mw_array**2) / np.sum(mw_array))
    PDI = Mw / Mn

    result["Mn"] = Mn
    result["Mw"] = Mw
    result["PDI"] = PDI

print("Statistics:")
for r in results:
    print(f"  - {r['name']}:")
    print(f"    Mn = {r['Mn']:.0f} g/mol, Mw = {r['Mw']:.0f} g/mol, PDI = {r['PDI']:.3f}")


## Step 7: Visualize Sampled vs Theoretical Distributions

We plot histograms of the sampled chains overlaid with the theoretical PDF (for continuous distributions) or PMF (for discrete distributions).

**Important:** 
- **Schulz–Zimm** is a continuous distribution in mass space (PDF). We plot `mass_pdf(M)` vs histogram of molecular weights.
- **Uniform, Poisson, Flory–Schulz** are discrete distributions in DP space (PMF). We plot `dp_pmf(n)` vs histogram of DPs.

Do not overlay a PMF on a PDF (different units).


In [ ]:
# Helper function: add statistics box to each subplot
def annotate_stats(ax, Mn: float, Mw: float, PDI: float, n_chains: int) -> None:
    """Add statistics box at top-right of axis."""
    Mn_txt = f"{Mn:.0f}\\,\\mathrm{{g/mol}}"
    Mw_txt = f"{Mw:.0f}\\,\\mathrm{{g/mol}}"

    txt = (
        rf"$M_n={Mn_txt}$" "\n"
        rf"$M_w={Mw_txt}$" "\n"
        rf"$\mathrm{{PDI}}={PDI:.3f}$" "\n"
        rf"$N={n_chains}$"
    )
    ax.text(
        0.96, 0.96, txt,
        transform=ax.transAxes,
        ha="right", va="top",
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="none", alpha=0.90),
    )


In [ ]:
# Create figure with 2x2 subplots
fig, axes = plt.subplots(
    2, 2,
    figsize=(5.5, 5.2),
    constrained_layout=True,
    sharex=False,
    sharey=False,
)
axes = axes.flatten()

for idx, (ax, result) in enumerate(zip(axes, results)):
    dist_obj = result["dist_obj"]

    if isinstance(dist_obj, SchulzZimmPolydisperse):
        # Continuous distribution in mass space (PDF)
        bins = 60
        n_grid = 700
        x_pad_lo, x_pad_hi = 0.3, 1.3

        mw = np.asarray(result["molecular_weights"], dtype=float)
        ax.hist(mw, bins=bins, density=True, histtype="step", label="Chain hist" if idx == 0 else "")

        M_min = max(0.0, mw.min() * x_pad_lo)
        M_max = mw.max() * x_pad_hi
        M_grid = np.linspace(M_min, M_max, n_grid)
        pdf = dist_obj.mass_pdf(M_grid)
        ax.plot(M_grid, pdf, label="Fit" if idx == 0 else "")

        Mn_val = result["Mn"]
        Mw_val = result["Mw"]
        pdf_Mn = float(dist_obj.mass_pdf(Mn_val))
        pdf_Mw = float(dist_obj.mass_pdf(Mw_val))
        ax.plot([Mn_val, Mn_val], [0, pdf_Mn], color="C2", linestyle="--")
        ax.plot([Mw_val, Mw_val], [0, pdf_Mw], color="C3", linestyle="--")
        
        # Add text labels for Mn and Mw
        ylim = ax.get_ylim()
        ax.text(Mn_val * 1.15, pdf_Mn * 1.05, f"$M_n$", color="C2", ha="center", va="bottom")
        ax.text(Mw_val * 1.1, pdf_Mw * 1.1, f"$M_w$", color="C3", ha="center", va="bottom")

        ax.set_xlabel(r"Molecular weight $M$ (g/mol)")
        ax.set_ylabel(r"PDF $f_M(M)$")

    elif isinstance(dist_obj, UniformPolydisperse):
        # Discrete distribution in DP space (PMF)
        dp = np.asarray(result["dps"], dtype=int)
        dp_min, dp_max = int(dp.min()), int(dp.max())
        bins_edges = np.arange(dp_min - 0.5, dp_max + 1.5, 1.0)
        ax.hist(dp, bins=bins_edges, density=True, histtype="step", label="Chain hist" if idx == 0 else "")

        support = np.arange(dp_min, dp_max + 1)
        pmf = dist_obj.dp_pmf(support)
        ax.plot(support, pmf, marker="o", label="Fit" if idx == 0 else "")

        # Calculate DP values for Mn and Mw
        monomer_masses = list(result["monomer_masses"].values())
        avg_monomer_mass = np.mean(monomer_masses)
        dp_Mn = result["Mn"] / avg_monomer_mass
        dp_Mw = result["Mw"] / avg_monomer_mass
        
        # Find closest DP in support for plotting
        dp_Mn_idx = np.argmin(np.abs(support - dp_Mn))
        dp_Mw_idx = np.argmin(np.abs(support - dp_Mw))
        dp_Mn_plot = support[dp_Mn_idx]
        dp_Mw_plot = support[dp_Mw_idx]
        pmf_Mn = pmf[dp_Mn_idx]
        pmf_Mw = pmf[dp_Mw_idx]
        
        ax.plot([dp_Mn_plot, dp_Mn_plot], [0, pmf_Mn], color="C2", linestyle="--")
        ax.plot([dp_Mw_plot, dp_Mw_plot], [0, pmf_Mw], color="C3", linestyle="--")
        ax.text(dp_Mn_plot * 1.1, pmf_Mn * 1.1, f"$M_n$", color="C2", ha="center", va="bottom")
        ax.text(dp_Mw_plot * 1.1, pmf_Mw * 1.1, f"$M_w$", color="C3", ha="center", va="bottom")

        ax.set_xlabel(r"Degree of polymerization $n$")
        ax.set_ylabel(r"PMF $P(n)$")
        ax.set_ylim(0, pmf.max() * 1.5)

    elif isinstance(dist_obj, PoissonPolydisperse):
        # Discrete distribution in DP space (PMF)
        dp = np.asarray(result["dps"], dtype=int)
        dp_min, dp_max = int(dp.min()), int(dp.max())
        bins_edges = np.arange(dp_min - 0.5, dp_max + 1.5, 1.0)
        ax.hist(dp, bins=bins_edges, density=True, histtype="step", label="Chain hist" if idx == 0 else "")

        theory_min = 1
        theory_max = max(dp_max, int(dp.mean() + 4 * dp.std()))
        support = np.arange(theory_min, theory_max + 1)
        pmf = dist_obj.dp_pmf(support)
        ax.plot(support, pmf, marker="o", label="Fit" if idx == 0 else "")

        # Calculate DP values for Mn and Mw
        monomer_masses = list(result["monomer_masses"].values())
        avg_monomer_mass = np.mean(monomer_masses)
        dp_Mn = result["Mn"] / avg_monomer_mass
        dp_Mw = result["Mw"] / avg_monomer_mass
        
        # Find closest DP in support for plotting
        dp_Mn_idx = np.argmin(np.abs(support - dp_Mn))
        dp_Mw_idx = np.argmin(np.abs(support - dp_Mw))
        dp_Mn_plot = support[dp_Mn_idx]
        dp_Mw_plot = support[dp_Mw_idx]
        pmf_Mn = pmf[dp_Mn_idx]
        pmf_Mw = pmf[dp_Mw_idx]
        
        ax.plot([dp_Mn_plot, dp_Mn_plot], [0, pmf_Mn], color="C2", linestyle="--")
        ax.plot([dp_Mw_plot, dp_Mw_plot], [0, pmf_Mw], color="C3", linestyle="--")
        ax.text(dp_Mn_plot * 1.1, pmf_Mn * 1.1, f"$M_n$", color="C2", ha="center", va="bottom")
        ax.text(dp_Mw_plot * 1.1, pmf_Mw * 1.1, f"$M_w$", color="C3", ha="center", va="bottom")

        ax.set_xlabel(r"Degree of polymerization $n$")
        ax.set_ylabel(r"PMF $P(n)$")

    elif isinstance(dist_obj, FlorySchulzPolydisperse):
        # Discrete distribution in DP space (PMF)
        dp = np.asarray(result["dps"], dtype=int)
        dp_min, dp_max = int(dp.min()), int(dp.max())
        bins_edges = np.arange(dp_min - 0.5, dp_max + 1.5, 1.0)
        ax.hist(dp, bins=bins_edges, density=True, histtype="step", label="Chain hist" if idx == 0 else "")

        theory_min = 1
        theory_max = max(dp_max, int(dp.mean() + 4 * dp.std()))
        support = np.arange(theory_min, theory_max + 1)
        pmf = dist_obj.dp_pmf(support)
        ax.plot(support, pmf, marker="o", label="Fit" if idx == 0 else "")

        # Calculate DP values for Mn and Mw
        monomer_masses = list(result["monomer_masses"].values())
        avg_monomer_mass = np.mean(monomer_masses)
        dp_Mn = result["Mn"] / avg_monomer_mass
        dp_Mw = result["Mw"] / avg_monomer_mass
        
        # Find closest DP in support for plotting
        dp_Mn_idx = np.argmin(np.abs(support - dp_Mn))
        dp_Mw_idx = np.argmin(np.abs(support - dp_Mw))
        dp_Mn_plot = support[dp_Mn_idx]
        dp_Mw_plot = support[dp_Mw_idx]
        pmf_Mn = pmf[dp_Mn_idx]
        pmf_Mw = pmf[dp_Mw_idx]
        
        ax.plot([dp_Mn_plot, dp_Mn_plot], [0, pmf_Mn], color="C2", linestyle="--")
        ax.plot([dp_Mw_plot, dp_Mw_plot], [0, pmf_Mw], color="C3", linestyle="--")
        ax.text(dp_Mn_plot * 1.1, pmf_Mn * 1.1, f"$M_n$", color="C2", ha="center", va="bottom")
        ax.text(dp_Mw_plot * 1.1, pmf_Mw * 1.1, f"$M_w$", color="C3", ha="center", va="bottom")

        ax.set_xlabel(r"Degree of polymerization $n$")
        ax.set_ylabel(r"PMF $P(n)$")
        
    else:
        raise TypeError(f"Unsupported distribution type: {type(dist_obj)}")

    ax.set_title(result["name"])
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    annotate_stats(ax, result["Mn"], result["Mw"], result["PDI"], result["n_chains"])
    if idx == 0:
        ax.legend(loc="best")

plt.show()


## Interpretation

- **PDF vs PMF**: use `mass_pdf(M)` for continuous mass distributions (Schulz–Zimm), and `dp_pmf(n)` for discrete DP distributions (Uniform/Poisson/Flory–Schulz). Do not overlay a PMF on a PDF (different units).
- **Target total mass**: a large `target_mass` generates many chains. For quick smoke tests, reduce the `target_total_mass` or use a smaller value in the GBigSMILES string.
- **Reproducibility**: this example fixes both the distribution `random_seed` and the Python `Random(...)` seed so runs are repeatable.

## Extensions

- **Turn a system plan into an atomistic structure**: feed `SystemPlan.chains` into a `PolymerBuilder` (atomistic build layer), then pack with Packmol to create an initial configuration.
- **Fit to experimental distributions**: implement a custom distribution that provides `mass_pdf/sample_mass` and plug it into the same pipeline.
- **Control copolymer composition**: adjust `WeightedSequenceGenerator` weights to change the repeat-unit composition.

## Summary

You now have the standard path for generating a polydisperse polymer **system plan** with MolPy:

- `parse_gbigsmiles` parses repeat units and `DistributionIR` from a GBigSMILES string
- `create_polydisperse_from_ir` turns `DistributionIR` into a samplable distribution (DP space or mass space)
- `PolydisperseChainGenerator` samples chain size and generates sequences + chain mass
- `SystemPlanner` accumulates chains until the target total mass is reached
- Finally, compute $M_n$ / $M_w$ / PDI from the sampled chains and visualize histogram vs theoretical PMF/PDF

To convert this chain plan into an atomistic system suitable for LAMMPS, continue with the atomistic build + packing guides.

# Extension: Build LAMMPS-Ready System from Sampled Chains

Now that we have a polydisperse chain ensemble, let's extend one distribution (Schulz-Zimm) to build a complete LAMMPS-ready system:

1. **Build 3D monomer library**: Convert repeat units to atomistic structures with 3D coordinates
2. **Build atomistic chains**: Convert chain sequences to 3D polymer structures
3. **Typify**: Assign OPLS-AA force field types
4. **Pack**: Pack chains into a periodic simulation box
5. **Export**: Write LAMMPS data and force field files

We'll use the **Schulz-Zimm** distribution from above as an example.


## Step 8: Build 3D Monomer Library

We need to convert the 2D repeat units to 3D atomistic structures with reactive ports for polymerization.


In [ ]:
# Select Schulz-Zimm distribution from results
schulz_result = next(r for r in results if r["name"] == "Schulz-Zimm")
schulz_chains = schulz_result["chain_generator"]
schulz_planner = SystemPlanner(
    chain_generator=schulz_result["chain_generator"],
    target_total_mass=schulz_result.get("target_mass", 1e7),
    max_rel_error=0.02,
)
schulz_system_plan = schulz_planner.plan_system(Random(random_seed))
schulz_chains_list = schulz_system_plan.chains[:10]  # Use first 10 chains for demo

# Get the original GBigSMILES IR to extract repeat units
schulz_gbigsmiles = next(d["gbigsmiles"] for d in _build_distributions() if d["name"] == "Schulz-Zimm")
schulz_ir = mp.parser.parse_gbigsmiles(schulz_gbigsmiles)
schulz_component_ir = schulz_ir.molecules[0]
schulz_mol_ir = schulz_component_ir.molecule
schulz_repeat_units = schulz_mol_ir.structure.stochastic_objects[0].repeat_units
schulz_stoch_obj = schulz_mol_ir.structure.stochastic_objects[0]

# Build 3D monomer library
from molpy.parser.smiles.converter import create_monomer_from_repeat_unit
from molpy.adapter.rdkit import RDKitAdapter
from molpy.compute.rdkit import Generate3D
from molpy.builder.polymer.port_utils import get_ports
from molpy.core.atomistic import Atomistic

schulz_monomer_structures_2d = []
for unit in schulz_repeat_units:
    monomer = create_monomer_from_repeat_unit(unit, schulz_stoch_obj)
    if monomer is not None:
        schulz_monomer_structures_2d.append(monomer)

# Build 3D structures
library = {}
for i, monomer_2d in enumerate(schulz_monomer_structures_2d):
    label = f"M{i}"
    
    # Generate 3D coordinates
    adapter = RDKitAdapter(internal=monomer_2d)
    generate_3d = Generate3D(add_hydrogens=True, embed=True, optimize=True, update_internal=True)
    adapter = generate_3d(adapter)
    monomer_3d = adapter.get_internal()
    
    # Generate topology
    monomer_3d.get_topo(gen_angle=True, gen_dihe=True)
    
    library[label] = monomer_3d

print(f"✅ Built {len(library)} monomers:")
for label, monomer in library.items():
    print(f"   {label}: {len(monomer.atoms)} atoms")


## Step 9: Build Atomistic Polymer Chains

Convert chain sequences to 3D atomistic structures using `PolymerBuilder`.


In [ ]:
# Set up polymer builder
from molpy.builder.polymer import PolymerBuilder
from molpy.builder.polymer.connectors import ReacterConnector
from molpy.builder.polymer.placer import Placer, CovalentSeparator, LinearOrienter
from molpy.reacter import (
    select_port,
    select_c_neighbor,
    select_hydroxyl_group,
    select_hydroxyl_h_only,
    form_single_bond,
)
from molpy.reacter.base import Reacter

reacter = Reacter(
    name="dehydration",
    anchor_selector_left=select_c_neighbor,
    anchor_selector_right=select_port,
    leaving_selector_left=select_hydroxyl_group,
    leaving_selector_right=select_hydroxyl_h_only,
    bond_former=form_single_bond,
)

# Create port_map
port_map = {}
monomer_labels = list(library.keys())
for left_label in monomer_labels:
    for right_label in monomer_labels:
        left_monomer = library[left_label]
        right_monomer = library[right_label]
        
        left_ports = get_ports(left_monomer)
        right_ports = get_ports(right_monomer)
        
        if ">" in left_ports and "<" in right_ports:
            port_map[(left_label, right_label)] = (">", "<")
        elif "<" in left_ports and ">" in right_ports:
            port_map[(left_label, right_label)] = ("<", ">")
        elif left_ports and right_ports:
            port_map[(left_label, right_label)] = (list(left_ports.keys())[0], list(right_ports.keys())[0])

connector = ReacterConnector(default=reacter, port_map=port_map)
separator = CovalentSeparator(buffer=0.0)
orienter = LinearOrienter()
placer = Placer(separator=separator, orienter=orienter)

builder = PolymerBuilder(
    library=library,
    connector=connector,
    placer=placer,
    typifier=None,  # Will typify later
)

# Build atomistic chains
atomistic_chains = []
print(f"Building {len(schulz_chains_list)} atomistic chains...")

for i, chain in enumerate(schulz_chains_list):
    # Convert sequence to CGSmiles
    sequence_labels = [f"[#M{idx}]" for idx in chain.monomers]
    cgsmiles = "{" + "".join(sequence_labels) + "}"
    
    # Build polymer
    result = builder.build(cgsmiles)
    polymer = result.polymer
    
    # Regenerate topology
    polymer.get_topo(gen_angle=True, gen_dihe=True)
    
    atomistic_chains.append(polymer)
    
    if (i + 1) % 5 == 0:
        print(f"   Built {i+1}/{len(schulz_chains_list)} chains...")

print(f"✅ Built {len(atomistic_chains)} atomistic chains")
if atomistic_chains:
    print(f"   Example chain: {len(atomistic_chains[0].atoms)} atoms, {len(atomistic_chains[0].bonds)} bonds")


## Step 10: Typify with Force Field

Assign LAMMPS atom types using OPLS-AA force field.


In [ ]:
# Load force field
from molpy.typifier.atomistic import OplsAtomisticTypifier

forcefield_path = "oplsaa.xml"
ff = mp.io.read_xml_forcefield(forcefield_path)
typifier = OplsAtomisticTypifier(ff, strict_typing=False)

print("Typifying chains...")
for i, polymer in enumerate(atomistic_chains):
    typifier.typify(polymer)
    if (i + 1) % 5 == 0:
        print(f"   Typified {i+1}/{len(atomistic_chains)} chains...")

print(f"✅ Typified {len(atomistic_chains)} chains")


## Step 11: Pack into Simulation Box

Use Packmol to pack chains into a periodic box with target density.


In [ ]:
# Calculate box size from target density
def calculate_box_size(chains: list[Atomistic], target_density: float = 1.0) -> float:
    """Calculate box length (Å) from target density (g/cm³)."""
    total_mw = 0.0
    for chain in chains:
        for atom in chain.atoms:
            symbol = atom.get("symbol") or atom.get("element", "C")
            elem = Element(symbol.upper())
            total_mw += elem.mass
    
    NA = 6.022e23
    total_mass_g = total_mw / NA
    volume_cm3 = total_mass_g / target_density
    volume_angstrom3 = volume_cm3 * 1e24
    box_length = volume_angstrom3 ** (1.0 / 3.0)
    return box_length

target_density = 0.01  # g/cm³ (low density for demo)
box_length = calculate_box_size(atomistic_chains, target_density)

print(f"✅ Calculated box size: {box_length:.2f} Å")
print(f"   Target density: {target_density} g/cm³")
print(f"   Number of chains: {len(atomistic_chains)}")

# Prepare frames for packing
from pathlib import Path
from molpy.pack import InsideBoxConstraint, Molpack
from molpy.core.box import Box

chain_frames = []
for polymer in atomistic_chains:
    frame = polymer.to_frame()
    # Normalize charge field
    if "charge" in frame["atoms"]:
        frame["atoms"]["q"] = frame["atoms"]["charge"]
    chain_frames.append(frame)

# Pack with Molpack
output_dir = Path("user-guide-output") / "05_polymer_polydisperse_lammps"
output_dir.mkdir(parents=True, exist_ok=True)

pack_workdir = output_dir / "packmol_work"
packer = Molpack(workdir=pack_workdir)

origin = np.array([0.0, 0.0, 0.0])
length = np.array([box_length, box_length, box_length])
box_constraint = InsideBoxConstraint(length=length, origin=origin)

print("\nPacking chains...")
for frame in chain_frames:
    packer.add_target(frame, number=1, constraint=box_constraint)

packed_frame = packer.optimize(max_steps=10000, seed=42)
print(f"✅ Packed system: {packed_frame['atoms'].nrows} atoms")

# Add box metadata
box = Box.cubic(length=box_length)
packed_frame.metadata["box"] = box

# Ensure charge fields exist
if "charge" in packed_frame["atoms"] and "q" not in packed_frame["atoms"]:
    packed_frame["atoms"]["q"] = packed_frame["atoms"]["charge"]
elif "q" in packed_frame["atoms"] and "charge" not in packed_frame["atoms"]:
    packed_frame["atoms"]["charge"] = packed_frame["atoms"]["q"]


## Step 12: Export to LAMMPS

Collect all unique types and write LAMMPS data and force field files.


In [ ]:
# Collect types
def collect_types_from_frames(*frames) -> dict[str, set[str]]:
    """Collect all type names from multiple frames."""
    atom_types = set()
    bond_types = set()
    angle_types = set()
    dihedral_types = set()
    
    for frame in frames:
        if "atoms" in frame and "type" in frame["atoms"]:
            for atom_type in frame["atoms"]["type"]:
                if atom_type:
                    type_str = str(atom_type)
                    if not type_str.isdigit():
                        atom_types.add(type_str)
        
        if "bonds" in frame and "type" in frame["bonds"]:
            for bond_type in frame["bonds"]["type"]:
                if bond_type:
                    type_str = str(bond_type)
                    if not type_str.isdigit():
                        bond_types.add(type_str)
        
        if "angles" in frame and "type" in frame["angles"]:
            for angle_type in frame["angles"]["type"]:
                if angle_type:
                    type_str = str(angle_type)
                    if not type_str.isdigit():
                        angle_types.add(type_str)
        
        if "dihedrals" in frame and "type" in frame["dihedrals"]:
            for dihedral_type in frame["dihedrals"]["type"]:
                if dihedral_type:
                    type_str = str(dihedral_type)
                    if not type_str.isdigit():
                        dihedral_types.add(type_str)
    
    return {
        "atom_types": atom_types,
        "bond_types": bond_types,
        "angle_types": angle_types,
        "dihedral_types": dihedral_types,
    }

types_from_config = collect_types_from_frames(packed_frame)

packed_frame.metadata["type_labels"] = {
    "atom_types": sorted(list(types_from_config["atom_types"])),
    "bond_types": sorted(list(types_from_config["bond_types"])),
    "angle_types": sorted(list(types_from_config["angle_types"])),
    "dihedral_types": sorted(list(types_from_config["dihedral_types"])),
}

print(f"\n✅ Collected types:")
print(f"   Atom types: {len(types_from_config['atom_types'])}")
print(f"   Bond types: {len(types_from_config['bond_types'])}")
print(f"   Angle types: {len(types_from_config['angle_types'])}")
print(f"   Dihedral types: {len(types_from_config['dihedral_types'])}")

# Write force field file
from molpy.io.data.lammps import LammpsDataWriter
from molpy.io.forcefield.lammps import LAMMPSForceFieldWriter

ff_path = output_dir / "polydisperse_system.ff"
ff_writer = LAMMPSForceFieldWriter(ff_path)
ff_writer.write(
    ff,
    atom_types=types_from_config["atom_types"],
    bond_types=types_from_config["bond_types"],
    angle_types=types_from_config["angle_types"],
    dihedral_types=types_from_config["dihedral_types"],
)

# Write data file
data_path = output_dir / "polydisperse_system.data"
data_writer = LammpsDataWriter(data_path, atom_style="full")
data_writer.write(packed_frame)

print(f"\n✅ Written LAMMPS files:")
print(f"   Force field: {ff_path}")
print(f"   Data file: {data_path}")


In [ ]:
# Compute statistics for script header
schulz_mw_array = np.array([chain.mass for chain in schulz_chains_list])
schulz_Mn = float(np.mean(schulz_mw_array))
schulz_Mw = float(np.sum(schulz_mw_array**2) / np.sum(schulz_mw_array))
schulz_PDI = schulz_Mw / schulz_Mn

script_content = f"""# LAMMPS input script for polydisperse polymer system
# Force field: OPLS-AA
# Distribution: Schulz-Zimm (Mn={schulz_Mn:.0f}, Mw={schulz_Mw:.0f}, PDI={schulz_PDI:.3f})
# Number of chains: {len(atomistic_chains)}
#
units           real
atom_style      full
boundary        p p p
dimension       3

read_data       polydisperse_system.data
include         polydisperse_system.ff
kspace_style    pppm 1.0e-5
special_bonds   lj/coul 0.0 0.0 0.5

neighbor        2.0 bin
neigh_modify    delay 0 every 1 check yes

print "=========================================="
print "Step 1: Energy Minimization"
print "=========================================="
minimize        1.0e-4 1.0e-4 1000 10000

print "=========================================="
print "Step 2: NVT Equilibration"
print "=========================================="
velocity        all create 300.0 12345
timestep        1.0
fix             nvt all nvt temp 300.0 300.0 100.0
thermo          100
thermo_style    custom step temp pe ke etotal press vol density
dump            1 all custom 1000 equil_nvt.lammpstrj id mol type x y z
run             10000
undump          1
unfix           nvt

print "=========================================="
print "Step 3: NPT Equilibration"
print "=========================================="
fix             npt all npt temp 300.0 300.0 100.0 iso 1.0 1.0 1000.0
dump            2 all custom 1000 equil_npt.lammpstrj id mol type x y z
run             10000
undump          2
unfix           npt

print "=========================================="
print "Step 4: Production MD"
print "=========================================="
fix             npt_prod all npt temp 300.0 300.0 100.0 iso 1.0 1.0 1000.0
dump            3 all custom 1000 production.lammpstrj id mol type xu yu zu
run             50000
undump          3
unfix           npt_prod

write_data      final_system.data
write_restart   final_system.restart

print "=========================================="
print "Simulation completed!"
print "=========================================="
"""

script_path = output_dir / "run_polydisperse.in"
with script_path.open("w") as f:
    f.write(script_content)

print(f"✅ Generated LAMMPS input script: {script_path}")


## Summary: From Distribution to LAMMPS

This extension demonstrated the complete workflow from a polydisperse distribution to a LAMMPS-ready system:

1. ✅ **System planning**: Used the sampled chain ensemble from Step 5 (Schulz-Zimm distribution)
2. ✅ **Monomer library**: Built 3D atomistic monomers with reactive ports
3. ✅ **Chain building**: Converted sequences to atomistic polymers via `PolymerBuilder`
4. ✅ **Typifying**: Assigned OPLS-AA force field types
5. ✅ **Packing**: Packed chains into periodic box with Packmol
6. ✅ **Export**: Generated LAMMPS data, force field, and input files

### Generated Files

All files are in `user-guide-output/05_polymer_polydisperse_lammps/`:

- `polydisperse_system.data` - LAMMPS data file (atom coordinates, topology)
- `polydisperse_system.ff` - Force field parameters
- `run_polydisperse.in` - LAMMPS input script

### Running the Simulation

```bash
cd user-guide-output/05_polymer_polydisperse_lammps
lmp -in run_polydisperse.in
```

This workflow can be applied to any of the four distributions (Schulz-Zimm, Uniform, Poisson, Flory-Schulz) by selecting a different result from the `results` list above.
